**Block 1 — Imports + device**

In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


**Block 2 — Model configuration**

In [38]:
class ModelArgs:
    vocab_size = 32000

    dim = 256
    n_layers = 4
    n_heads = 8
    n_kv_heads = 2
    hidden_dim = 1024

    max_seq_len = 512
    norm_eps = 1e-5


args = ModelArgs()

print("D:", args.dim)
print("Heads:", args.n_heads)
print("Head dim:", args.dim // args.n_heads)
print("FFN dim:", args.hidden_dim)

D: 256
Heads: 8
Head dim: 32
FFN dim: 1024


**Block 3 — RMSNorm**

[B,S,256]

     ↓
     
[B,S,1]  -->  RMS value

     ↓
    
[B,S,256]

nn.Module
   │
   
   ├── nn.Linear
   
   ├── nn.Conv2d
   
   ├── nn.Embedding
   
   ├── Attention ← ours
   
   ├── FeedForward     ← ours
   
   ├── RMSNorm         ← ours
   
   └── LlamaModel      ← ours

In [39]:
class RMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim)) #scaling

    def forward(self, x):
        # x: [B,S,D]
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        x_norm = x / rms
        return self.weight * x_norm

**Block 4 — Precompute RoPE**

In [40]:
def precompute_rope(head_dim,max_seq_len,theta=10000.0):
    # 0,2,4,...30
    indices = torch.arange(0,head_dim,2).float()
    freqs = 1.0 / (theta ** (indices / head_dim))
    # [max_seq_len]
    positions = torch.arange( max_seq_len).float()
    angles = torch.outer( positions, freqs )
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    return cos, sin

**Block 5 — Apply RoPE**

In [41]:
def apply_rope(x, cos, sin):
    # x: [B,H,S,Hd]
    # 32 → 16 + 16
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    # [S,16] --> [1,1,S,16]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    # Rotate pairs
    new_even = ( x_even * cos - x_odd * sin )
    new_odd = ( x_even * sin + x_odd * cos)

    # [B,H,S,16] + [B,H,S,16] -->[B,H,S,16,2]
    out = torch.stack( [new_even, new_odd],dim=-1)
    # 16 × 2 → 32 [B,H,S,16,2] → [B,H,S,32]
    return out.flatten(-2)

**Block 6 — Group-Head Attention**

In [42]:
def repeat_kv(x, n_rep):
    B, S, n_kv_heads, head_dim = x.shape

    return (
        x[:, :, :, None, :]
        .expand(B, S, n_kv_heads, n_rep, head_dim)
        .reshape(B, S, n_kv_heads * n_rep, head_dim)
    )

In [43]:
class Attention(nn.Module):

    def __init__(self, args):
        super().__init__()
        self.dim = args.dim
        self.n_heads = args.n_heads
        self.n_kv_heads = args.n_kv_heads
        self.head_dim = (args.dim // args.n_heads) # Number of Q heads sharing one KV head
        self.n_rep = self.n_heads // self.n_kv_heads
        # Q K V
        self.wq = nn.Linear(args.dim,self.n_heads * self.head_dim,bias=False) #Q will have 8 heads
        
        self.wk = nn.Linear(args.dim,self.n_kv_heads * self.head_dim,bias=False) #K has 2 heads

        self.wv = nn.Linear( args.dim,self.n_kv_heads * self.head_dim,bias=False) #V has 2 heads

        self.wo = nn.Linear( args.dim,args.dim,bias=False)

    def forward(self, x, cos, sin):
        # x = [B,S,D]
        B, S, D = x.shape
        # Q K V
        q = self.wq(x)
        k = self.wk(x)
        v = self.wv(x)
        # [B,S,256] --> Q--[B,S,8,32] K, V --[B,S,2,32]
        q = q.view( B, S, self.n_heads, self.head_dim )
        k = k.view(B, S,self.n_kv_heads,self.head_dim)
        v = v.view( B, S, self.n_kv_heads, self.head_dim )
       # [B,S,H,Hd] → [B,H,S,Hd]
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

     # RoPE for current sequence
        cos = cos[:S]
        sin = sin[:S]

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        # GQA: repeat K/V HEADS
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        # Attention scores
        # [B,H,S,Hd] @ [B,H,Hd,S] → [B,H,S,S]
        scores = torch.matmul(q,k.transpose(-2, -1))
        scores = scores/ math.sqrt(self.head_dim)
        # Causal mask
        mask = torch.triu(torch.ones(S,S,device=x.device,dtype=torch.bool),diagonal=1)
        scores = scores.masked_fill(mask,float("-inf"))
        # Softmax
        attention_weights = F.softmax(scores,dim=-1)
        # Weighted values
        # [B,H,S,S] @ [B,H,S,Hd] → [B,H,S,Hd]
        out = torch.matmul(attention_weights,v)
        # [B,H,S,Hd]→ [B,S,H,Hd]
        out = out.transpose(1, 2)
        # [B,S,H,Hd] → [B,S,D]
        out = out.contiguous().view(B,S,self.dim)
        # [B,S,D] → [B,S,D]
        return self.wo(out)

**Block 7 — SwiGLU FFN**

In [44]:
class FeedForward(nn.Module):

    def __init__(self, args):
        super().__init__()
        self.w1 = nn.Linear( args.dim, args.hidden_dim,bias=False )
        # 1024 → 256
        self.w2 = nn.Linear(args.hidden_dim,args.dim,bias=False)
        # 256 → 1024
        self.w3 = nn.Linear( args.dim, args.hidden_dim, bias=False)
    def forward(self, x):
        # Gate branch
        a = self.w1(x)
        a = F.silu(a)
        # Value branch
        b = self.w3(x)
        # SwiGLU gating
        gated = a * b
        # Return to model dimension
        out = self.w2(gated)
        return out

**Block 8 — Transformer Block**
normed_x  = norm(x)

attn_out  = attention(normed_x)

h         = x + attn_out


normed_h  = norm(h)

ffn_out   = ffn(normed_h)

out       = h + ffn_out

In [45]:
class TransformerBlock(nn.Module):
    def __init__(self, args):
        super().__init__()
        self.attention = Attention(args)
        self.feed_forward = FeedForward(args)
        self.attention_norm = RMSNorm(args.dim,args.norm_eps) #rms norm before attention
        self.ffn_norm = RMSNorm(args.dim,args.norm_eps)    #rms norm before swiglu
    def forward(self, x, cos, sin):
        # Attention
        normed_x = self.attention_norm(x)
        attention_out = self.attention(normed_x,cos,sin)
        # Residual #1
        h = x + attention_out
        # SwiGLU
        normed_h = self.ffn_norm(h)
        ffn_out = self.feed_forward(normed_h)
        # Residual #2
        out = h + ffn_out
        return out

x
↓
Norm
↓
Attention
↓
ADD x
↓
Norm
↓
SwiGLU
↓
ADD
↓
output

**Block 9 — Complete LLaMA model**

TOKEN IDs [B,S]
      ↓
TOKEN EMBEDDING
      ↓
[B,S,D]
      ↓
┌────────────────────────────┐
│     TRANSFORMER BLOCK      │
│                            │
│ RMSNorm                    │
│    ↓                       │
│ Attention                  │
│   ├─ Q,K,V                 │
│   ├─ RoPE on Q,K           │
│   ├─ MHA                   │
│   └─ Output projection     │
│    ↓                       │
│ + Residual                 │
│    ↓                       │
│ RMSNorm                    │
│    ↓                       │
│ SwiGLU / FFN               │
│    ↓                       │
│ + Residual                 │
└────────────────────────────┘
      ↓
repeat × n_layers
      ↓
FINAL RMSNorm
      ↓
LM HEAD
      ↓
LOGITS [B,S,V]

In [46]:
class LlamaModel(nn.Module):

    def __init__(self, args):
        super().__init__()

        self.args = args

        # Embedding
        self.tok_embeddings = nn.Embedding(args.vocab_size,args.dim)
        # Transformer blocks
        self.layers = nn.ModuleList([
            TransformerBlock(args)
            for _ in range(args.n_layers)
        ])
        # Final RMSNorm
        self.norm = RMSNorm(args.dim,args.norm_eps )
        self.output = nn.Linear(args.dim,args.vocab_size,bias=False )
        # Precompute RoPE
        cos, sin = precompute_rope(args.dim // args.n_heads,args.max_seq_len) #prepare precompute rope
        self.register_buffer("cos", cos)                                      #store it inside buffer ,means cos, sin = precompute_rope(32, 512)
        self.register_buffer("sin",sin)
    def forward(self, tokens):
        # tokens = [B,S]
        B, S = tokens.shape
        if S > self.args.max_seq_len:
            raise ValueError(f"Sequence length {S} exceeds ",f"max_seq_len={self.args.max_seq_len}")
        # Embedding
        # [B,S]→[B,S,D]
        x = self.tok_embeddings(tokens)
        # RoPE for current S
        cos = self.cos[:S]
        sin = self.sin[:S]
        # Transformer blocks
        for layer in self.layers:
            x = layer(x,cos,sin)
        # Final norm
        x = self.norm(x)
        # Vocabulary logits
        # [B,S,D] →[B,S,V]
        logits = self.output(x)
        return logits

**Block 10 — Test the entire model**

In [47]:
args = ModelArgs()

model = LlamaModel(args).to(device)

tokens = torch.randint(
    0,
    args.vocab_size,
    (2, 5),
    device=device
)

with torch.no_grad():
    logits = model(tokens)

print("Tokens :", tokens.shape)
print("Logits :", logits.shape)

Tokens : torch.Size([2, 5])
Logits : torch.Size([2, 5, 32000])


**Block 11 — Download Tiny Shakespeare**

In [48]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

print("Dataset downloaded!")

Dataset downloaded!


**Block 12 — Load the text**

In [49]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Total characters:", len(text))

print("\nFirst 500 characters:\n")
print(text[:500])

Total characters: 1115394

First 500 characters:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


**Block 13 — Character Tokenizer**

In [50]:

chars = sorted(list(set(text))) #set for the sake of getting unique chars in text
vocab_size = len(chars) # 2. Vocabulary size
print("Vocabulary size:", vocab_size)
print("Characters:", chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)} # . Character → integer
idx_to_char = {i: ch for i, ch in enumerate(chars)}  # int---> char 
def encode(s):
    return [char_to_idx[ch] for ch in s] # Encode text → token IDs
def decode(ids):
    return "".join([idx_to_char[i] for i in ids]) #. Decode token IDs → text

Vocabulary size: 65
Characters: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


**Block 14 — Encode Entire Dataset + Split It**

In [51]:
# Convert entire Shakespeare text into token IDs
data = torch.tensor(encode(text),dtype = torch.long)

# 90% training, 10% validation
split_idx = int(0.9 * len(data))

train_data = data[:split_idx]
val_data = data[split_idx:]

print("Full data shape :", data.shape)
print("Train shape     :", train_data.shape)
print("Val shape       :", val_data.shape)

Full data shape : torch.Size([1115394])
Train shape     : torch.Size([1003854])
Val shape       : torch.Size([111540])


**BLOCK 15 —CREATE TRAINING BATCHES**

In [52]:
batch_size = 16
seq_len = 128
def get_batch(data):
    starts = torch.randint( 0,len(data) - seq_len - 1,(batch_size,))
    # Inputs
    x = torch.stack([data[i : i + seq_len]for i in starts])
    # Targets = same sequence shifted 1 token forward
    y = torch.stack([data[i + 1 : i + seq_len + 1]for i in starts ])
    return x.to(device), y.to(device)

**BLOCK 16 — CONFIGURE MODEL FOR REAL DATA**

In [53]:
args = ModelArgs()
args.vocab_size = vocab_size # Use the ACTUAL Shakespeare vocabulary size
args.max_seq_len = seq_len # Our training sequence length
model = LlamaModel(args).to(device) # Create a fresh model
print("Vocabulary size :", args.vocab_size)
print("Max seq length  :", args.max_seq_len)
total_params = sum(p.numel() for p in model.parameters())
print("Total parameters:", total_params)

Vocabulary size : 65
Max seq length  : 128
Total parameters: 3836672


**BLOCK 17 — TRAINING**

In [54]:
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)
num_steps = 10000
model.train()
for step in range(num_steps):
    x, y = get_batch(train_data)   #  Get training batch
    logits = model(x)              #  Forward pass
    loss = F.cross_entropy(logits.reshape(-1, vocab_size),y.reshape(-1))    # Calculate loss
    optimizer.zero_grad()       # Clear old gradients
    loss.backward()               # Backpropagation
    optimizer.step()              #  Update LLaMA weights

    # Print progress
    if step % 100 == 0:
        print(f"Step {step} | Loss: {loss.item():.4f}")

Step 0 | Loss: 4.3485
Step 100 | Loss: 2.3423
Step 200 | Loss: 2.0564
Step 300 | Loss: 1.7784
Step 400 | Loss: 1.6609
Step 500 | Loss: 1.5930
Step 600 | Loss: 1.5011
Step 700 | Loss: 1.5780
Step 800 | Loss: 1.4594
Step 900 | Loss: 1.4654
Step 1000 | Loss: 1.4043
Step 1100 | Loss: 1.3932
Step 1200 | Loss: 1.4302
Step 1300 | Loss: 1.3591
Step 1400 | Loss: 1.4012
Step 1500 | Loss: 1.3653
Step 1600 | Loss: 1.3732
Step 1700 | Loss: 1.3055
Step 1800 | Loss: 1.2448
Step 1900 | Loss: 1.3449
Step 2000 | Loss: 1.2734
Step 2100 | Loss: 1.3712
Step 2200 | Loss: 1.2846
Step 2300 | Loss: 1.2866
Step 2400 | Loss: 1.3381
Step 2500 | Loss: 1.2648
Step 2600 | Loss: 1.3569
Step 2700 | Loss: 1.2439
Step 2800 | Loss: 1.3043
Step 2900 | Loss: 1.2798
Step 3000 | Loss: 1.2000
Step 3100 | Loss: 1.2394
Step 3200 | Loss: 1.2609
Step 3300 | Loss: 1.2010
Step 3400 | Loss: 1.2191
Step 3500 | Loss: 1.2044
Step 3600 | Loss: 1.2178
Step 3700 | Loss: 1.1604
Step 3800 | Loss: 1.2118
Step 3900 | Loss: 1.1911
Step 4000 | 

STEP 1
→ take one batch of 16 sequences
→ forward
→ loss
→ backward
→ update weights

STEP 2
→ take another batch of 16 sequences
→ forward
→ loss
→ backward
→ update weights

...

**Block 18 — Generate Text From Your LLaMA**

In [55]:
@torch.no_grad()
def generate(model, start_text, max_new_tokens=300):
    model.eval()
    tokens = torch.tensor(encode(start_text),dtype=torch.long,device=device).unsqueeze(0)                  # [S] → [1,S] llama expects b,s
    for _ in range(max_new_tokens):
        input_tokens = tokens[:, -args.max_seq_len:] # Keep only what fits inside our model
        logits = model(input_tokens)   # Forward pass
        next_token_logits = logits[:, -1, :]    # Only care about prediction after LAST token
        probs = F.softmax(next_token_logits, dim=-1)  # Convert logits → probabilities
        next_token = torch.multinomial(probs,num_samples=1) # Sample next character
        tokens = torch.cat([tokens, next_token],dim=1) # Add predicted character to sequence
    return decode(tokens[0].tolist())

**Block 19 — Validation Loss**

In [56]:
model.eval()

val_losses = []
with torch.no_grad():
    for _ in range(100):
        x, y = get_batch(val_data)  # Get validation batch
        logits = model(x) #  # Forward pass
        loss = F.cross_entropy(logits.reshape(-1, vocab_size),y.reshape(-1) ) ## Calculate validation loss
        val_losses.append(loss.item())
avg_val_loss = sum(val_losses) / len(val_losses)

print(f"Validation Loss: {avg_val_loss:.4f}")

Validation Loss: 1.9855


In [57]:
prompt = "ROMEO:"

generated = generate(model,start_text=prompt,max_new_tokens=500)
print(generated)

ROMEO:
So, sometimes cannot this young modest girl: our knowledge
h not a lonted and at iddeed from consul:
How canst thou, or thou beest, Romeo:
There lie shall, as yours, perforce hath appear,
Which I'll figght again, methought, death,
Still cold again, and true Romeo;
This is the duke's of their perfection oQnwaring,
That will she, brave Bolingbroke, can her son,
But stirn you get thine here's some belonging;
If your dishonest for blunt on this night
Earth-kindled by your blushes, of all within!
Ab
